In [45]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/rllm")

import json

from pathlib import Path
from rllm.data import DatasetRegistry, Dataset
from rllm.eval.agent_loader import load_agent, register_agent, list_agents
from rllm.runner import Runner
from rllm.types import AgentConfig, Task
from rllm.eval.evaluator_loader import load_evaluator
from tqdm import tqdm
from utils.mpr import MultipleProcessRunnerSimplifier

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Tokenizer check

In [40]:
from transformers import AutoTokenizer

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "你好",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "你好",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]

model_name = "/sujin/Models/Qwen/Qwen3-4B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "嘿嘿嘿\n哈哈哈哈"},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
    tools=TOOLS
)

print(text)

<|im_start|>system
嘿嘿嘿
哈哈哈哈

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "calculate", "description": "你好", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'"}}, "required": ["expression"]}}}
{"type": "function", "function": {"name": "calculate", "description": "你好", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'"}}, "required": ["expression"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Give me a sh

# Single test

In [8]:
dataset = DatasetRegistry.load_dataset("math500", "test")

item = dataset[0]
print(item.keys())   # → dict_keys(['question', 'answer', ...)
print(item["question"])

dict_keys(['question', 'ground_truth', 'data_source'])
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$


In [6]:
register_agent("math_tool_agent", "cookbooks.math_tool_agent.sync_math_tool_agent:math_tool_agent")
list_agents()

[{'name': 'math',
  'source': 'registered',
  'description': '',
  'module': 'cookbooks.math.sync_math_flow:math_flow'},
 {'name': 'math_tool_agent',
  'source': 'registered',
  'description': '',
  'module': 'cookbooks.math_tool_agent.sync_math_tool_agent:math_tool_agent'},
 {'name': 'bash',
  'source': 'built-in',
  'description': 'Multi-turn ReAct bash loop inside a sandbox.',
  'module': 'rllm.harnesses.bash.BashHarness'},
 {'name': 'claude-code',
  'source': 'built-in',
  'description': 'Run the Claude Code CLI inside the sandbox.',
  'module': 'rllm.harnesses.claude_code.ClaudeCodeHarness'},
 {'name': 'react',
  'source': 'built-in',
  'description': 'One-shot LLM call. Default for data tasks (math, MCQ, QA).',
  'module': 'rllm.harnesses.react.ReActHarness'}]

In [47]:
agent = load_agent("math_tool_agent")

task = Task(
    id="math500",
    instruction=item["question"],
    # instruction="列出你所有可用的工具以及他们的描述给我",
    metadata=item,
    dataset_dir=Path("/root/.rllm/datasets/math500"),  # 相对路径
)

config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="notebook-0",
    )

episode = agent.run(task, config)

In [51]:
print(len(episode.trajectories[0].steps))
print(json.dumps(episode.trajectories[0].steps[1].chat_completions, indent=4))

text = tokenizer.apply_chat_template(
    episode.trajectories[0].steps[-1].chat_completions,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
    tools=TOOLS
)

print(text)

2
[
    {
        "role": "system",
        "content": "You are a math assistant that solves competition math problems step by step.\nYou have access to a calculator tool and you MUST use it.\n\nIMPORTANT rules you must follow:\n1. You MUST call the calculator tool at least once before giving your final answer. Answers given without any prior tool call will be marked wrong.\n2. Do NOT perform arithmetic in your head \u2014 every computation must go through the calculator.\n3. Break the problem into small steps. Make one tool call per step, then reason about the result.\n4. When you have the final answer, put it in \\boxed{ANSWER} in your response.\n"
    },
    {
        "role": "user",
        "content": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$"
    },
    {
        "role": "assistant",
        "content": "<think>\nOkay, so I need to convert the rectangular coo

In [3]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")
# client = OpenAI(base_url="http://172.98.0.90:37863/v1", api_key="EMPTY")

messages: list[dict] = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "你好"},
]

resp = client.chat.completions.create(
    model="/sujin/Models/Qwen/Qwen3-4B",
    messages=messages,
    temperature=0.6,
    max_tokens=1000,
    max_completion_tokens=1000,
    timeout=100,
)
content = resp.choices[0].message.content or ""
print(content)

<think>
嗯，用户发来“你好”，这是一条非常简单的问候语。首先，我需要回应这个问候，保持友好和专业的态度。用户可能只是想开始对话，或者有更具体的问题需要解决。我需要确定他们需要什么帮助，可能是在询问某个问题，或者只是需要一个简单的回应。

接下来，我应该考虑用户的潜在需求。他们可能对某个话题感兴趣，或者遇到了问题需要帮助。例如，他们可能在询问技术问题、生活建议，或者只是想进行闲聊。作为AI助手，我需要保持开放和包容的态度，鼓励用户进一步说明他们的需求。

同时，我需要确保回应简洁明了，避免使用过于复杂的语言。用户可能希望得到快速而直接的回答，所以需要在回复中提供足够的信息，同时保持友好。另外，我需要检查是否有任何可能的误解，比如用户可能用“你好”作为测试，或者有其他隐藏的意图。

最后，我应该保持积极的态度，让用户感到被重视和支持。通过友好的问候和开放的提问，可以促进更深入的对话，帮助用户更好地解决问题或获得所需的信息。
</think>

你好！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我能够帮助你回答问题、创作内容、提供建议等。无论你有什么需求，都可以告诉我，我会尽力协助你！😊 你今天想聊什么？
